In [ ]:
import random
import numpy as np
import torch
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from agent import FloodWarningEnv

SEED = 2
N_EVAL_EPISODES = 1000
MODEL_PATH = "./models/lstm_vecnorm/best_model"
VECNORM_PATH = "./models/lstm_vecnorm/vec_normalize.pkl"

# Reward weights
FALSE_WEIGHT = 0.301511
MISSED_WEIGHT = 1.007163
JUMP_WEIGHT = 0.878312

def make_env():
    return lambda: FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT)


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env = make_vec_env(make_env(), n_envs=1, seed=SEED)
env = VecNormalize.load(VECNORM_PATH, env)
env.training = False
env.norm_reward = False

model = RecurrentPPO.load(MODEL_PATH, env=env)

episode_rewards = []
for ep in range(N_EVAL_EPISODES):
    obs = env.reset()
    lstm_states = None
    episode_start = np.ones((1,), dtype=bool)
    total_reward = 0.0
    done = False

    while not done:
        action, lstm_states = model.predict(
            obs,
            state=lstm_states,
            episode_start=episode_start,
            deterministic=True,
        )
        obs, reward, done, _ = env.step(action)
        episode_start = done
        total_reward += reward[0]

    episode_rewards.append(total_reward)
    print(f"Episode {ep + 1:>3}/{N_EVAL_EPISODES}  reward: {total_reward:.4f}")

print(f"\nMean reward : {np.mean(episode_rewards):.4f}")
print(f"Std reward  : {np.std(episode_rewards):.4f}")
print(f"Min reward  : {np.min(episode_rewards):.4f}")
print(f"Max reward  : {np.max(episode_rewards):.4f}")

env.close()

In [ ]:
import random
import numpy as np
import pickle
import math

from baselines.threshold_baseline import ThresholdAgent
from agent import FloodWarningEnv

SEED = 2
N_EVAL_EPISODES = 1000

# Reward weights
FALSE_WEIGHT = 0.301511
MISSED_WEIGHT = 1.007163
JUMP_WEIGHT = 0.878312

SEASON_TO_IDX = {"Spring": 0, "Summer": 1, "Autumn": 2, "Winter": 3}

def encode_obs(raw_obs: dict) -> dict:
    obs = dict(raw_obs)
    idx = SEASON_TO_IDX[obs["season"]]
    obs["season_sin"] = math.sin(2 * math.pi * idx / 4)
    obs["season_cos"] = math.cos(2 * math.pi * idx / 4)
    del obs["season"]
    return obs

random.seed(SEED)
np.random.seed(SEED)

# Load the saved decision tree model
with open("baselines/dt_model.pkl", "rb") as f:
    clf = pickle.load(f)

agent = ThresholdAgent(clf)

episode_rewards = []
for ep in range(N_EVAL_EPISODES):
    env = FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT)
    env.env.sample_features()
    env.env.update_derived()
    total_reward = 0.0

    for step in range(200):  # MAX_STEPS
        raw_obs = env.env.get_observable_features()
        obs = encode_obs(raw_obs)
        action = agent.predict(obs)

        _, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    episode_rewards.append(total_reward)
    print(f"Episode {ep + 1:>3}/{N_EVAL_EPISODES}  reward: {total_reward:.4f}")

print(f"\nMean reward : {np.mean(episode_rewards):.4f}")
print(f"Std reward  : {np.std(episode_rewards):.4f}")
print(f"Min reward  : {np.min(episode_rewards):.4f}")
print(f"Max reward  : {np.max(episode_rewards):.4f}")

env.close()

In [ ]:
import random
import numpy as np

from baselines.random_baseline import RandomAgent
from agent import FloodWarningEnv

SEED = 2
N_EVAL_EPISODES = 1000

# Reward weights
FALSE_WEIGHT = 0.301511
MISSED_WEIGHT = 1.007163
JUMP_WEIGHT = 0.878312

random.seed(SEED)
np.random.seed(SEED)

agent = RandomAgent()

episode_rewards = []

for ep in range(N_EVAL_EPISODES):
    env = FloodWarningEnv(
        false_weight=FALSE_WEIGHT,
        missed_weight=MISSED_WEIGHT,
        jump_weight=JUMP_WEIGHT
    )
    env.env.sample_features()
    env.env.update_derived()
    total_reward = 0.0

    for step in range(200):  # MAX_STEPS
        # Random agent does not use observation
        action = agent.predict()
        _, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    episode_rewards.append(total_reward)
    print(f"Episode {ep + 1:>3}/{N_EVAL_EPISODES}  reward: {total_reward:.4f}")

print(f"\nMean reward : {np.mean(episode_rewards):.4f}")
print(f"Std reward  : {np.std(episode_rewards):.4f}")
print(f"Min reward  : {np.min(episode_rewards):.4f}")
print(f"Max reward  : {np.max(episode_rewards):.4f}")

env.close()

In [ ]:
import random
import numpy as np
import pickle
import torch
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv

from agent import FloodWarningEnv
from baselines.threshold_baseline import ThresholdAgent
from baselines.random_baseline import RandomAgent

SEED = 2
N_EVAL_EPISODES = 10

# Reward weights
FALSE_WEIGHT = 0.301511
MISSED_WEIGHT = 1.007163
JUMP_WEIGHT = 0.878312

# -----------------------------
# Load agents
# -----------------------------

# Threshold agent
with open("baselines/dt_model.pkl", "rb") as f:
    dt_clf = pickle.load(f)
threshold_agent = ThresholdAgent(dt_clf)

# Random agent
random_agent = RandomAgent()

# RL agent
MODEL_PATH = "./models/lstm_vecnorm/best_model"
VECNORM_PATH = "./models/lstm_vecnorm/vec_normalize.pkl"

def make_env():
    return lambda: FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

vec_env = DummyVecEnv([make_env()])
vec_env = VecNormalize.load(VECNORM_PATH, vec_env)
vec_env.training = False
vec_env.norm_reward = False

rl_model = RecurrentPPO.load(MODEL_PATH, env=vec_env)

# -----------------------------
# Evaluation
# -----------------------------

rewards_threshold = []
rewards_random = []
rewards_rl = []

for ep in range(N_EVAL_EPISODES):
    # -----------------------------
    # Sample initial environment state for Threshold/Random
    # -----------------------------
    init_env = FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT)
    init_env.env.sample_features()
    init_env.env.update_derived()
    init_state = init_env.env  # Save environment object to clone for other agents

    # -----------------------------
    # Threshold agent
    # -----------------------------
    env_thr = FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT)
    env_thr.env = init_state  # Start from same initial sampled environment
    total_reward = 0.0
    done = False
    for _ in range(200):
        obs = env_thr._get_obs()
        action = threshold_agent.predict(dict(
            precipitation=obs[0],
            flood_depth=obs[1],
            season_sin=obs[2],
            season_cos=obs[3],
            holiday=obs[4],
            soil_moisture=obs[5],
            water_density=obs[6],
            water_distance=obs[7],
            elevation=obs[8],
            impervious_surface=obs[9],
            historical_flood_flag=int(obs[10]),
            deprivation_index=obs[11],
            residential=obs[12],
            commercial=obs[13],
            industrial=obs[14],
            agriculture=obs[15],
            transport=obs[16],
            population_density=obs[17]
        ))
        obs_next, reward, done, truncated, _ = env_thr.step(action)
        total_reward += reward
        if done or truncated:
            break
    rewards_threshold.append(total_reward)
    env_thr.close()

    # -----------------------------
    # Random agent
    # -----------------------------
    env_rand = FloodWarningEnv(false_weight=FALSE_WEIGHT, missed_weight=MISSED_WEIGHT, jump_weight=JUMP_WEIGHT)
    env_rand.env = init_state  # Start from same initial environment
    total_reward = 0.0
    done = False
    for _ in range(200):
        obs = env_rand._get_obs()
        action = random_agent.predict()
        obs_next, reward, done, truncated, _ = env_rand.step(action)
        total_reward += reward
        if done or truncated:
            break
    rewards_random.append(total_reward)
    env_rand.close()

    # -----------------------------
    # RL agent
    # -----------------------------
    obs_rl = vec_env.reset()
    lstm_states = None
    episode_start = np.ones((1,), dtype=bool)
    total_reward = 0.0
    done = False
    for _ in range(200):
        action, lstm_states = rl_model.predict(
            obs_rl,
            state=lstm_states,
            episode_start=episode_start,
            deterministic=True
        )
        obs_rl, reward, done, _ = vec_env.step(action)
        total_reward += reward[0]
        episode_start = done
        if done[0]:
            break
    rewards_rl.append(total_reward)

    print(f"Episode {ep+1:>3} | Threshold: {rewards_threshold[-1]:.2f} | Random: {rewards_random[-1]:.2f} | RL: {rewards_rl[-1]:.2f}")

# -----------------------------
# Summary statistics
# -----------------------------
for name, rewards in zip(["Threshold", "Random", "RL"], [rewards_threshold, rewards_random, rewards_rl]):
    print(f"{name} agent -> mean: {np.mean(rewards):.4f}, std: {np.std(rewards):.4f}, min: {np.min(rewards):.4f}, max: {np.max(rewards):.4f}")